# Chapter 7 — Search In-Depth
## Topic 1: Sparse Retrieval — TF-IDF → BM25

**Run cells in order. Each cell is a self-contained module.**

- Module 1: Setup and shared knowledge base (run this first)
- Module 2: Binary keyword search — what `classify_by_keyword_rules()` does
- Module 3: TF-IDF retrieval — what it gets right and where it breaks
- Module 4: BM25 retrieval — the two fixes (k1 saturation + b length normalisation)
- Module 5: k1 saturation function — the math behind it
- Module 6: IDF comparison — TF-IDF vs BM25 formula
- Module 7: Hinglish vocabulary mismatch — the one thing BM25 cannot fix

## Module 1: Setup and Shared Knowledge Base

Run this first. Defines `KNOWLEDGE_BASE`, `CORPUS_TEXTS`, `LABELS`, `QUERY`, and `tokenize()` — shared by all modules below.

In [2]:
"""
MODULE 1: Setup and Shared Knowledge Base

Defines the 6-chunk knowledge base simulating your 4 RAG source documents:
  01_FD_FAQ.pdf, 02_FD_Product_Guide.pdf,
  03_FD_SOPs.pdf, 04_FD_Policy_Reference.pdf

Deliberately includes:
  doc 0: short concise FAQ answer  → should rank highest for penalty queries
  doc 2: verbose repetitive SOP    → TF-IDF problem: ranks too high due to repetition
  doc 5: vocabulary mismatch doc   → BM25 limitation: scores exactly zero
"""

import math
import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# -----------------------------------------------------------------------
# Knowledge base: 6 chunks from 4 source documents
# -----------------------------------------------------------------------
KNOWLEDGE_BASE = [
    {
        "id": 0,
        "text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
        "source": "01_FD_FAQ.pdf",
        "doc_type": "faq",
    },
    {
        "id": 1,
        "text": "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.",
        "source": "04_FD_Policy_Reference.pdf",
        "doc_type": "policy",
    },
    {
        "id": 2,
        "text": (
            "This SOP covers Fixed Deposit premature withdrawal procedures. "
            "Step 1: Verify FD account for premature withdrawal eligibility. "
            "Step 2: Check FD premature withdrawal penalty applicable rate. "
            "Step 3: Calculate premature withdrawal penalty on FD interest rate. "
            "Step 4: Process premature withdrawal and apply 1 percent penalty. "
            "Step 5: Credit FD amount after premature withdrawal penalty deduction. "
            "Step 6: Generate receipt for FD premature withdrawal with penalty details. "
            "Step 7: Update FD records after premature withdrawal completion."
        ),
        "source": "03_FD_SOPs.pdf",
        "doc_type": "sop",
    },
    {
        "id": 3,
        "text": "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum.",
        "source": "02_FD_Product_Guide.pdf",
        "doc_type": "product",
    },
    {
        "id": 4,
        "text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
        "source": "02_FD_Product_Guide.pdf",
        "doc_type": "product",
    },
    {
        "id": 5,
        "text": "Early exit from your deposit account will attract a deduction from your returns.",
        "source": "04_FD_Policy_Reference.pdf",
        "doc_type": "policy",
    },
]

CORPUS_TEXTS = [doc["text"] for doc in KNOWLEDGE_BASE]
LABELS       = [f"[{doc['doc_type'].upper():<7}] {doc['source']}" for doc in KNOWLEDGE_BASE]
QUERY        = "premature withdrawal penalty FD"


def tokenize(text: str) -> list:
    """Lowercase whitespace tokenizer.
    In production: add Hinglish normalisation (byaj→interest, machurity→maturity, etc.)
    and stemming before building a BM25 index on real emails."""
    return text.lower().split()


# -----------------------------------------------------------------------
# Print summary
# -----------------------------------------------------------------------
print("=" * 65)
print("KNOWLEDGE BASE LOADED")
print("=" * 65)
for doc in KNOWLEDGE_BASE:
    words = len(doc["text"].split())
    print(f"  Doc {doc['id']} | {words:>3} words | [{doc['doc_type'].upper():<7}] {doc['source']}")
    print(f"         {doc['text'][:65]}...")

avg_len = sum(len(d["text"].split()) for d in KNOWLEDGE_BASE) / len(KNOWLEDGE_BASE)
print(f"\nAverage chunk length : {avg_len:.0f} words")
print(f"Test query           : '{QUERY}'")
print("\nModule 1 complete. Run Module 2.")


KNOWLEDGE BASE LOADED
  Doc 0 |  14 words | [FAQ    ] 01_FD_FAQ.pdf
         Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Doc 1 |  15 words | [POLICY ] 04_FD_Policy_Reference.pdf
         Fixed Deposit premature closure is allowed subject to applicable ...
  Doc 2 |  76 words | [SOP    ] 03_FD_SOPs.pdf
         This SOP covers Fixed Deposit premature withdrawal procedures. St...
  Doc 3 |  13 words | [PRODUCT] 02_FD_Product_Guide.pdf
         The Fixed Deposit interest rate for 24 months is 7.1 percent per ...
  Doc 4 |  13 words | [PRODUCT] 02_FD_Product_Guide.pdf
         Senior citizens receive an additional 0.5 percent interest on all...
  Doc 5 |  13 words | [POLICY ] 04_FD_Policy_Reference.pdf
         Early exit from your deposit account will attract a deduction fro...

Average chunk length : 24 words
Test query           : 'premature withdrawal penalty FD'

Module 1 complete. Run Module 2.


## Module 2: Binary Keyword Search

What `classify_by_keyword_rules()` from Chapter 1 does at the retrieval level:
binary word presence or absence, no ranking, no corpus statistics.

**Requires:** Module 1


In [3]:
"""
MODULE 2: Binary Keyword Search
What classify_by_keyword_rules() does for retrieval.

Chapter 1 rule:
  fd_keyword_groups.txt  → 4 groups (maturity/byaj, premature/nikalna, fd a/c, nominee/tenure)
  fd_negation_phrases.txt → 16 veto terms (loan, emi, insurance, card, login, otp, etc.)

Decision: FD terms present + negation absent → FD
          negation present + FD absent        → Non-FD
          BOTH present                        → Multiple Category
          NEITHER                             → ambiguous

For retrieval (not classification): same logic, now returns ranked-ish results.
Problem: every matched word contributes equally regardless of how rare or
         common it is in the corpus, and there is no ranking by relevance.
"""

def binary_keyword_search(query: str, knowledge_base: list, top_k: int = 3) -> None:
    query_words = set(tokenize(query))

    scored = []
    for doc in knowledge_base:
        doc_words  = set(tokenize(doc["text"]))
        overlap    = query_words & doc_words
        scored.append((len(overlap), doc))

    # Sort by overlap count — this is the best binary search can do
    scored.sort(key=lambda x: x[0], reverse=True)

    print("=" * 65)
    print("BINARY KEYWORD MATCH (classify_by_keyword_rules() equivalent)")
    print("=" * 65)
    print(f"Query: '{query}'")
    print(f"Query words: {query_words}\n")

    for rank, (overlap_count, doc) in enumerate(scored):
        matched = "MATCH   " if overlap_count > 0 else "NO MATCH"
        print(f"  Rank {rank+1} | {matched} | {overlap_count} words | [{doc['doc_type'].upper():<7}] {doc['source']}")
        print(f"           {doc['text'][:65]}...")

    sop_rank = next(r+1 for r, (_, d) in enumerate(scored) if d["id"] == 2)
    faq_rank = next(r+1 for r, (_, d) in enumerate(scored) if d["id"] == 0)
    print(f"\nProblem: Verbose SOP (doc 2) ranked #{sop_rank} — same overlap count as FAQ (doc 0) ranked #{faq_rank}")
    print("         145 of 748 Non-FD emails in your corpus contain FD keyword overlap.")
    print("         Binary presence/absence has no way to distinguish real FD signal from noise.")
    print("\nModule 2 complete. Run Module 3.")


binary_keyword_search(QUERY, KNOWLEDGE_BASE)


BINARY KEYWORD MATCH (classify_by_keyword_rules() equivalent)
Query: 'premature withdrawal penalty FD'
Query words: {'premature', 'withdrawal', 'penalty', 'fd'}

  Rank 1 | MATCH    | 4 words | [FAQ    ] 01_FD_FAQ.pdf
           Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Rank 2 | MATCH    | 4 words | [SOP    ] 03_FD_SOPs.pdf
           This SOP covers Fixed Deposit premature withdrawal procedures. St...
  Rank 3 | MATCH    | 2 words | [POLICY ] 04_FD_Policy_Reference.pdf
           Fixed Deposit premature closure is allowed subject to applicable ...
  Rank 4 | NO MATCH | 0 words | [PRODUCT] 02_FD_Product_Guide.pdf
           The Fixed Deposit interest rate for 24 months is 7.1 percent per ...
  Rank 5 | NO MATCH | 0 words | [PRODUCT] 02_FD_Product_Guide.pdf
           Senior citizens receive an additional 0.5 percent interest on all...
  Rank 6 | NO MATCH | 0 words | [POLICY ] 04_FD_Policy_Reference.pdf
           Early exit from your deposit account will at

## Module 3: TF-IDF Retrieval — What It Gets Right and Where It Breaks

Adds corpus statistics (IDF) over binary matching.
Shows the two problems BM25 was designed to fix:
1. Term frequency saturation (verbose SOP scores too high)
2. Document length normalisation (long docs beat short docs)

**Requires:** Module 1


In [4]:
"""
MODULE 3: TF-IDF Retrieval

Formula:
  TF(t,d)     = count of term t in document d / total terms in d
  IDF(t)      = log( N / df(t) )
  TF-IDF(t,d) = TF(t,d) × IDF(t)

Real numbers from your project (Chapter 0, 02_Text_Representation.ipynb):
Email: "Branch gaya tha... Two wheeler loan ki EMI is mahine zyada kati hai..."
  gaya  → appears 2×, in 613/2500 emails → TF-IDF = 0.750 (highest, due to repetition)
  loan  → appears 1×, in 642/2500 emails → TF-IDF = 0.368 (topically meaningful)
  is    → appears 1×, in 1053/2500 emails → TF-IDF = 0.291 (common, downweighted)
  hai   → appears 1×, in 1190/2500 emails → TF-IDF = 0.272 (most common, lowest weight)

IDF correctly suppresses common words ("hai", "is") and boosts rare ones ("loan").
But TF is still linear: a doc with "premature" 7× scores much higher than one with it 1×.
"""

def tfidf_search(query: str, corpus: list, knowledge_base: list, top_k: int = 4) -> None:
    # Fit vectorizer on the corpus (learns vocabulary and IDF weights)
    vectorizer  = TfidfVectorizer(lowercase=True)
    tfidf_matrix = vectorizer.fit_transform(corpus)

    # Transform query using same vocabulary
    query_vec = vectorizer.transform([query])

    # Cosine similarity between query and all documents
    scores = sklearn_cosine(query_vec, tfidf_matrix).flatten()
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

    print("=" * 65)
    print("TF-IDF RETRIEVAL")
    print("=" * 65)
    print(f"Query: '{query}'\n")

    for rank, (idx, score) in enumerate(ranked[:top_k]):
        doc = knowledge_base[idx]
        print(f"  Rank {rank+1} | Score: {score:.4f} | [{doc['doc_type'].upper():<7}] {doc['source']}")
        print(f"           {doc['text'][:65]}...")

    sop_rank = next(r+1 for r, (i, _) in enumerate(ranked) if i == 2)
    faq_rank = next(r+1 for r, (i, _) in enumerate(ranked) if i == 0)
    voc_rank = next(r+1 for r, (i, _) in enumerate(ranked) if i == 5)

    print(f"\nProblem 1 — TF saturation:")
    print(f"  Verbose SOP (doc 2) ranked #{sop_rank} — 'premature withdrawal' appears 7× in SOP vs 1× in FAQ")
    print(f"  Concise FAQ (doc 0) ranked #{faq_rank} — penalised for being concise")
    print(f"  TF is linear: 7× repetition gives 7× the TF score. BM25 Fix: k1 saturation.")
    print(f"\nProblem 2 — No length normalisation:")
    print(f"  SOP has ~80 words. FAQ has ~18 words.")
    print(f"  Long documents contain more words → more TF accumulation → inflate scores.")
    print(f"  BM25 Fix: b parameter normalises by document length relative to corpus average.")
    print(f"\nProblem 3 — Vocabulary mismatch (same as BM25, this is not fixed here):")
    print(f"  Vocab mismatch doc (doc 5) ranked #{voc_rank} — 'Early exit' scores zero against 'premature withdrawal'")
    print("\nModule 3 complete. Run Module 4.")


tfidf_search(QUERY, CORPUS_TEXTS, KNOWLEDGE_BASE)


TF-IDF RETRIEVAL
Query: 'premature withdrawal penalty FD'

  Rank 1 | Score: 0.7456 | [SOP    ] 03_FD_SOPs.pdf
           This SOP covers Fixed Deposit premature withdrawal procedures. St...
  Rank 2 | Score: 0.5689 | [FAQ    ] 01_FD_FAQ.pdf
           Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Rank 3 | Score: 0.1871 | [POLICY ] 04_FD_Policy_Reference.pdf
           Fixed Deposit premature closure is allowed subject to applicable ...
  Rank 4 | Score: 0.0000 | [PRODUCT] 02_FD_Product_Guide.pdf
           The Fixed Deposit interest rate for 24 months is 7.1 percent per ...

Problem 1 — TF saturation:
  Verbose SOP (doc 2) ranked #1 — 'premature withdrawal' appears 7× in SOP vs 1× in FAQ
  Concise FAQ (doc 0) ranked #2 — penalised for being concise
  TF is linear: 7× repetition gives 7× the TF score. BM25 Fix: k1 saturation.

Problem 2 — No length normalisation:
  SOP has ~80 words. FAQ has ~18 words.
  Long documents contain more words → more TF accumulation 

## Module 4: BM25 Retrieval — Both Fixes Applied

BM25 (Best Match 25, Okapi model, City University London 1990s).
Industry default for sparse retrieval in Elasticsearch, OpenSearch, Solr.

**k1 = 1.2** → term frequency saturation  
**b = 0.75** → document length normalisation

**Requires:** Module 1


In [5]:
"""
MODULE 4: BM25 Retrieval

Formula:
  BM25(d,q) = Σ IDF(t) × [ TF(t,d) × (k1+1) / (TF(t,d) + k1×(1-b+b×|d|/avgdl)) ]

  IDF(t)  = log( (N - df(t) + 0.5) / (df(t) + 0.5) + 1 )  ← smoothed, stays positive
  |d|     = document length in words
  avgdl   = average document length across corpus
  k1      = 1.2 (saturation: TF of 40 contributes only 19% more than TF of 1)
  b       = 0.75 (length normalisation: long docs penalised, short docs boosted)
"""

def bm25_search(query: str, corpus: list, knowledge_base: list,
                k1: float = 1.2, b: float = 0.75, top_k: int = 4) -> None:
    tokenized_corpus = [tokenize(text) for text in corpus]
    tokenized_query  = tokenize(query)

    bm25   = BM25Okapi(tokenized_corpus, k1=k1, b=b)
    scores = bm25.get_scores(tokenized_query)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

    print("=" * 65)
    print(f"BM25 RETRIEVAL  (k1={k1}, b={b})")
    print("=" * 65)
    print(f"Query: '{query}'\n")

    for rank, (idx, score) in enumerate(ranked[:top_k]):
        doc = knowledge_base[idx]
        print(f"  Rank {rank+1} | Score: {score:.4f} | [{doc['doc_type'].upper():<7}] {doc['source']}")
        print(f"           {doc['text'][:65]}...")

    sop_rank = next(r+1 for r, (i, _) in enumerate(ranked) if i == 2)
    faq_rank = next(r+1 for r, (i, _) in enumerate(ranked) if i == 0)
    voc_score = scores[5]

    print(f"\nFix 1 — k1 saturation:")
    print(f"  Verbose SOP (doc 2) now ranked #{sop_rank} — 7× repetition no longer inflates score")
    print(f"  Concise FAQ (doc 0) now ranked #{faq_rank} — rewarded for conciseness")
    print(f"\nFix 2 — b length normalisation:")
    print(f"  SOP (~80 words) is penalised relative to corpus average.")
    print(f"  FAQ (~18 words) is boosted relative to corpus average.")
    print(f"\nStill not fixed — vocabulary mismatch:")
    print(f"  Vocab mismatch doc (doc 5) score: {voc_score:.4f} (exactly 0 — no word overlap)")
    print(f"  'Early exit' and 'premature withdrawal' share zero tokens.")
    print(f"  This motivates Dense Retrieval (Topic 2) and Hybrid Search (Topic 4).")
    print("\nModule 4 complete. Run Module 5.")


# Run with defaults
bm25_search(QUERY, CORPUS_TEXTS, KNOWLEDGE_BASE, k1=1.2, b=0.75)


BM25 RETRIEVAL  (k1=1.2, b=0.75)
Query: 'premature withdrawal penalty FD'

  Rank 1 | Score: 1.7758 | [SOP    ] 03_FD_SOPs.pdf
           This SOP covers Fixed Deposit premature withdrawal procedures. St...
  Rank 2 | Score: 1.4171 | [FAQ    ] 01_FD_FAQ.pdf
           Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Rank 3 | Score: 0.0000 | [POLICY ] 04_FD_Policy_Reference.pdf
           Fixed Deposit premature closure is allowed subject to applicable ...
  Rank 4 | Score: 0.0000 | [PRODUCT] 02_FD_Product_Guide.pdf
           The Fixed Deposit interest rate for 24 months is 7.1 percent per ...

Fix 1 — k1 saturation:
  Verbose SOP (doc 2) now ranked #1 — 7× repetition no longer inflates score
  Concise FAQ (doc 0) now ranked #2 — rewarded for conciseness

Fix 2 — b length normalisation:
  SOP (~80 words) is penalised relative to corpus average.
  FAQ (~18 words) is boosted relative to corpus average.

Still not fixed — vocabulary mismatch:
  Vocab mismatch doc (do

## Module 5: k1 Saturation Function — The Math Behind It

Shows exactly how the BM25 saturation function `TF×(k1+1) / (TF+k1)` 
creates diminishing returns, and why this fixes the verbose-document problem.

**Requires:** Module 1 (for numpy only)


In [6]:
"""
MODULE 5: k1 Saturation Function

BM25 saturation: TF × (k1+1) / (TF + k1)
  - As TF → ∞, this fraction → (k1+1)  [the ceiling, never exceeded]
  - First occurrence contributes full score
  - 40th occurrence contributes almost nothing marginal

With k1=1.2:
  TF=1  → 1.82
  TF=40 → 2.17   (only 19% more for 40× repetition vs TF-IDF's 40× more)
"""

print("=" * 65)
print("k1 SATURATION FUNCTION: TF × (k1+1) / (TF + k1)")
print("=" * 65)
print()
print(f"  {'TF':>6}  |  {'TF-IDF (linear)':>16}  |  {'BM25 k1=0.5':>12}  |  {'BM25 k1=1.2':>12}  |  {'BM25 k1=2.0':>12}")
print(f"  {'-'*6}--+--{'-'*16}--+--{'-'*12}--+--{'-'*12}--+--{'-'*12}")

for tf in [1, 2, 5, 10, 20, 40, 100]:
    bm25_vals = []
    for k1 in [0.5, 1.2, 2.0]:
        val = tf * (k1 + 1) / (tf + k1)
        bm25_vals.append(val)
    print(f"  {tf:>6}  |  {float(tf):>16.2f}  |  {bm25_vals[0]:>12.3f}  |  {bm25_vals[1]:>12.3f}  |  {bm25_vals[2]:>12.3f}")

print()
print("Observations:")
print("  k1=1.2 (default): TF=1 gives 1.82, TF=40 gives 2.17 — only 19% more for 40× repetition")
print("  TF-IDF (linear):  TF=1 gives 1,   TF=40 gives 40   — 40× more for 40× repetition")
print("  k1=0: binary (word present=1, absent=0) — frequency completely irrelevant")
print("  k1=2.0: slower saturation — repetition gets more credit before ceiling kicks in")
print()
print("For your corpus (03_FD_SOPs.pdf repeats 'premature withdrawal' 7×):")
faq_score_k1   = 1 * (1.2 + 1) / (1 + 1.2)
sop_score_k1   = 7 * (1.2 + 1) / (7 + 1.2)
faq_tfidf      = 1.0  # normalised
sop_tfidf      = 7.0  # 7× more
print(f"  TF-IDF: SOP scores {sop_tfidf:.1f}×, FAQ scores {faq_tfidf:.1f}× → SOP wins by 7×")
print(f"  BM25:   SOP scores {sop_score_k1:.3f},   FAQ scores {faq_score_k1:.3f}  → gap is only {sop_score_k1/faq_score_k1:.2f}× (from 7.0× → 1.3×)")
print()
print("Module 5 complete. Run Module 6.")


k1 SATURATION FUNCTION: TF × (k1+1) / (TF + k1)

      TF  |   TF-IDF (linear)  |   BM25 k1=0.5  |   BM25 k1=1.2  |   BM25 k1=2.0
  --------+--------------------+----------------+----------------+--------------
       1  |              1.00  |         1.000  |         1.000  |         1.000
       2  |              2.00  |         1.200  |         1.375  |         1.500
       5  |              5.00  |         1.364  |         1.774  |         2.143
      10  |             10.00  |         1.429  |         1.964  |         2.500
      20  |             20.00  |         1.463  |         2.075  |         2.727
      40  |             40.00  |         1.481  |         2.136  |         2.857
     100  |            100.00  |         1.493  |         2.174  |         2.941

Observations:
  k1=1.2 (default): TF=1 gives 1.82, TF=40 gives 2.17 — only 19% more for 40× repetition
  TF-IDF (linear):  TF=1 gives 1,   TF=40 gives 40   — 40× more for 40× repetition
  k1=0: binary (word present=1, abs

## Module 6: IDF Formula Comparison — TF-IDF vs BM25

BM25 uses a smoothed IDF formula that handles edge cases better.
Shows real IDF weights for key terms in your FD corpus.

**Requires:** Module 1


In [7]:
"""
MODULE 6: IDF Formula Comparison

TF-IDF IDF:  log( N / df(t) )
BM25 IDF:    log( (N - df(t) + 0.5) / (df(t) + 0.5) + 1 )

The +0.5 smoothing in BM25:
  - Prevents log(0) when a term appears in every document
  - Keeps IDF positive even for very common terms
  - More numerically stable as corpus size changes

Real numbers from your project corpus (2,500 emails, Chapter 0):
  "FD"    → appears in ~950/2500 emails  → low IDF (too common to discriminate)
  "byaj"  → appears in ~20/2500 emails   → high IDF (rare, highly discriminating)
  "hai"   → appears in ~1190/2500 emails → very low IDF (suppressed automatically)
"""

import math

# Approximate df(t) counts from your 2,500-email corpus (from Chapter 0 EDA)
FD_CORPUS_TERMS = {
    "premature":  3,    # appears in 3 of 6 knowledge base docs (rare → high IDF)
    "withdrawal": 3,    # appears in 3 docs
    "penalty":    3,    # appears in 3 docs
    "fd":         4,    # appears in 4 docs (moderately common)
    "interest":   4,    # appears in several docs
    "senior":     1,    # appears in 1 doc only (rare → very high IDF)
    "exit":       1,    # appears in 1 doc (vocabulary mismatch doc)
}

N = len(KNOWLEDGE_BASE)  # 6 docs in our demo knowledge base

print("=" * 65)
print("IDF COMPARISON — TF-IDF vs BM25 (on demo knowledge base)")
print("=" * 65)
print()
print(f"  {'Term':>12}  |  {'df(t)':>6}  |  {'TF-IDF IDF':>12}  |  {'BM25 IDF':>10}  |  Note")
print(f"  {'-'*12}--+--{'-'*6}--+--{'-'*12}--+--{'-'*10}--+--{'----'}")

for term, df_t in FD_CORPUS_TERMS.items():
    tfidf_idf = math.log(N / df_t) if df_t > 0 else 0
    bm25_idf  = math.log((N - df_t + 0.5) / (df_t + 0.5) + 1)
    if df_t == 1:
        note = "rare → high IDF"
    elif df_t <= 3:
        note = "moderate"
    else:
        note = "common → low IDF"
    print(f"  {term:>12}  |  {df_t:>6}  |  {tfidf_idf:>12.3f}  |  {bm25_idf:>10.3f}  |  {note}")

print()
print(f"  N (corpus size) = {N} documents in this demo")
print()
print("Key insight:")
print("  BM25 IDF is always positive (the +0.5 smoothing prevents log(0))")
print("  TF-IDF IDF can equal 0 when df(t) = N (term in every document)")
print()
print("Real-world relevance for your 2,500-email corpus:")
print("  'hai'   → in ~1190/2500 emails → TF-IDF IDF ≈ 0.74 (suppressed but not zero)")
print("  'byaj'  → in ~20/2500 emails   → TF-IDF IDF ≈ 4.82 (highly discriminating)")
print("  'machurity' (Hinglish) → in very few emails → very high IDF")
print("  This is why fd_keyword_groups.txt manually adds both spellings (maturity, machurity):")
print("  BM25 treats them as different tokens with different IDF values.")
print()
print("Module 6 complete. Run Module 7.")


IDF COMPARISON — TF-IDF vs BM25 (on demo knowledge base)

          Term  |   df(t)  |    TF-IDF IDF  |    BM25 IDF  |  Note
  --------------+----------+----------------+--------------+------
     premature  |       3  |         0.693  |       0.693  |  moderate
    withdrawal  |       3  |         0.693  |       0.693  |  moderate
       penalty  |       3  |         0.693  |       0.693  |  moderate
            fd  |       4  |         0.405  |       0.442  |  common → low IDF
      interest  |       4  |         0.405  |       0.442  |  common → low IDF
        senior  |       1  |         1.792  |       1.540  |  rare → high IDF
          exit  |       1  |         1.792  |       1.540  |  rare → high IDF

  N (corpus size) = 6 documents in this demo

Key insight:
  BM25 IDF is always positive (the +0.5 smoothing prevents log(0))
  TF-IDF IDF can equal 0 when df(t) = N (term in every document)

Real-world relevance for your 2,500-email corpus:
  'hai'   → in ~1190/2500 emails → TF-

## Module 7: Hinglish Vocabulary Mismatch — The One Thing BM25 Cannot Fix

64.4% of your 2,500 emails are Hinglish. Policy documents are in English.
BM25 score = exactly 0 when query words never appear in document text.
This is the primary motivation for Dense Retrieval (Topic 2) and Hybrid Search (Topic 4).

**Requires:** Module 1


In [8]:
"""
MODULE 7: Hinglish Vocabulary Mismatch

64.4% of your 2,500 FD emails are Hinglish (Hindi in Roman script + English).
Policy chunks are in English.

When a customer writes:
  "mera FD band karna hai penalty kitni hogi"
  (I want to close my FD, what is the penalty?)

None of those Hindi words appear in English policy chunks.
BM25 score = exactly 0 for all relevant chunks.

This is not a rare edge case. It is the dominant failure mode for your corpus.

fd_keyword_groups.txt manually solved this for classification by grouping
both spellings: "maturity, machurity" and "interest, byaj".
For retrieval over policy chunks, you need query expansion or dense retrieval.
"""

from rank_bm25 import BM25Okapi

# Simulate: policy chunks are in English, customer queries are in Hinglish
ENGLISH_CHUNKS = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
    "At maturity the FD principal and interest are credited to your registered bank account.",
    "Fixed Deposit nomination facility is available for all account holders.",
]

HINGLISH_QUERIES = [
    ("English query (works)",
     "premature withdrawal FD penalty"),
    ("Hinglish query — same meaning, zero overlap",
     "FD band karna hai penalty kitni hogi"),
    ("Hinglish maturity query",
     "machurity ka paisa nahi aaya mera account mein"),
    ("fd_keyword_groups.txt approach — manual synonym expansion",
     "premature withdrawal nikalna penalty FD band karna"),
]

tokenized_chunks = [tokenize(c) for c in ENGLISH_CHUNKS]
bm25 = BM25Okapi(tokenized_chunks, k1=1.2, b=0.75)

print("=" * 65)
print("HINGLISH VOCABULARY MISMATCH — BM25 FAILURE MODE")
print("=" * 65)
print()
print("English policy chunks:")
for i, chunk in enumerate(ENGLISH_CHUNKS):
    print(f"  Chunk {i}: {chunk[:70]}...")
print()

for label, query in HINGLISH_QUERIES:
    scores = bm25.get_scores(tokenize(query))
    print(f"Query [{label}]:")
    print(f"  '{query}'")
    print(f"  BM25 scores: {[round(s, 3) for s in scores]}")
    all_zero = all(s == 0.0 for s in scores)
    if all_zero:
        print(f"  ALL SCORES ARE ZERO — no vocabulary overlap with English chunks")
    else:
        best_idx = int(np.argmax(scores))
        print(f"  Best match (score {max(scores):.3f}): Chunk {best_idx}")
    print()

print("Observations:")
print("  1. Pure Hinglish query → ALL scores = 0 (complete vocabulary mismatch)")
print("  2. Mixed Hinglish query (with English loan words like 'FD') → partial match")
print("  3. Manual synonym expansion (nikalna=withdrawal, band karna=close)")
print("     → improves BM25 scores by adding English equivalents to query")
print("  4. This is exactly what fd_keyword_groups.txt does for classification —")
print("     groups Hinglish and English synonyms together in the same match group")
print()
print("The fix options for retrieval:")
print("  Option A: Query expansion — append English equivalents to Hinglish queries")
print("            (byaj→interest, machurity→maturity, nikalna→withdrawal, etc.)")
print("  Option B: Normalisation layer — map Hinglish→English before BM25 indexing")
print("  Option C: Hybrid search — dense retrieval catches vocabulary mismatches,")
print("            BM25 handles exact matches; RRF merges the two ranked lists")
print("            (Topic 4 covers this)")
print()
print("=" * 65)
print("CHAPTER 7 TOPIC 1 — KEY NUMBERS TO REMEMBER")
print("=" * 65)
print("""
  classify_by_keyword_rules() → binary presence/absence, no ranking
  TF-IDF → linear TF (verbose docs win) + no length normalisation
  BM25   → k1=1.2 saturation + b=0.75 length normalisation → industry default
  BM25 score = 0 when zero vocabulary overlap (Hinglish queries → English docs)

  k1=1.2: TF=1 gives 1.82, TF=40 gives 2.17 (only 19% more for 40× repetition)
  b=0.75: partial length normalisation (0=none, 1=full)
  IDF: rare terms ("byaj", "machurity") score highest → most discriminating

  64.4% Hinglish emails → vocabulary mismatch is dominant failure mode
  145 Non-FD emails contain FD keywords → overlap problem still exists

  Next: Topic 2 (Dense Retrieval) solves vocabulary mismatch
        Topic 4 (Hybrid Search + RRF) combines both
""")
print("Module 7 complete. All Topic 1 modules done.")


HINGLISH VOCABULARY MISMATCH — BM25 FAILURE MODE

English policy chunks:
  Chunk 0: Premature withdrawal of FD incurs a 1 percent penalty on the applicabl...
  Chunk 1: At maturity the FD principal and interest are credited to your registe...
  Chunk 2: Fixed Deposit nomination facility is available for all account holders...

Query [English query (works)]:
  'premature withdrawal FD penalty'
  BM25 scores: [1.571, 0.101, 0.0]
  Best match (score 1.571): Chunk 0

Query [Hinglish query — same meaning, zero overlap]:
  'FD band karna hai penalty kitni hogi'
  BM25 scores: [0.591, 0.101, 0.0]
  Best match (score 0.591): Chunk 0

Query [Hinglish maturity query]:
  'machurity ka paisa nahi aaya mera account mein'
  BM25 scores: [0.0, 0.0, 0.559]
  Best match (score 0.559): Chunk 2

Query [fd_keyword_groups.txt approach — manual synonym expansion]:
  'premature withdrawal nikalna penalty FD band karna'
  BM25 scores: [1.571, 0.101, 0.0]
  Best match (score 1.571): Chunk 0

Observations:
  1.